# Data4Good Case Challenge



![Data4Good](Data4Good.png)



## 📖 Background
Artificial Intelligence (AI) is rapidly transforming education by providing students with instant access to information and adaptive learning tools. Still, it also introduces significant risks, such as the spread of misinformation and fabricated content. Research indicates that large language models (LLMs) often confidently generate factually incorrect or “hallucinated” responses, which can mislead learners and erode trust in digital learning platforms.

The 4th Annual Data4Good Competition challenges participants to develop innovative analytics solutions to detect and improve factuality in AI-generated educational content, ensuring that AI advances knowledge rather than confusion.

## 💾 The data

The data provided is a Questions/Answer dataset to determine if the answer is factual, not factual (contradiction), or irrelevant to the question.


- Question: The question asked/prompted for
- Context: Relevant contextual support for the question
- Answer: The answer provided by an AI
- Type:  A categorical variable with three possible levels – Factual, Contradiction, Irrelevant:
  - Factual: the answer is correct
  - Contradiction: the answer is incorrect
  - Irrelevant: the answer has nothing to do with the question
  
There are 21,021 examples in the dataset (`data/train.json`) that you will experiment with.


The test dataset (`data/test.json`) contains 2000 examples that you predict as one of the three provided classes. In addition to classification performance we are seeking as detailed as possible methodologies of your step-by-step approach in your notebooks. Discuss what worked well, what did not work well, and your suggestions or ideas if a general approach to these types of problems might exist.


### Previewing the Training Data

Let's load and preview the `train.json` dataset to understand its structure and contents.

First, we need to mount the Google Drive to the right location so we have access to it

In [1]:
# from google.colab import drive
import pandas as pd
import json

# drive.mount('/content/drive')

In [3]:
# Load the train.json file
# data_path = "/content/drive/MyDrive/the bayes-ic instinct/data/train.json"
data_path = "./data/train.json"

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame
train_df = pd.DataFrame(data)

# Show the first 50 rows
# train_df.head(50)

# data_path = "/content/drive/MyDrive/the bayes-ic instinct/data/test.json"
# with open(data_path, "r", encoding="utf-8") as f:
#     data = json.load(f)
# test_df = pd.DataFrame(data)

Remember to update the `data_path` in the data loading cell to point to the correct location of `train.json` in your mounted Google Drive.

In [4]:
pd.set_option("display.max_colwidth", 1000)
train_df.head(5)

,answer,type,context,question
0,"In 1512, Parliament passed a significant act that resulted in the construction of fortifications in Plymouth.",factual,"During the Hundred Years' War a French attack (1340) burned a manor house and took some prisoners, but failed to get into the town. In 1403 the town was burned by Breton raiders. In the late fifteenth century a 'castle quadrate' was constructed close to the area now known as The Barbican; it included four round towers, one at each corner, as featured on the city coat of arms. The castle served to protect Sutton Pool, which is where the fleet was based in Plymouth prior to the establishment of Plymouth Dockyard. In 1512 an Act of Parliament was passed for further fortifying Plymouth, and a series of fortifications were then built, including defensive walls at the entrance to Sutton Pool (across which a chain would be extended in time of danger). Defences on St Nicholas Island also date from this time, and a string of six artillery blockhouses were built, including one on Fishers Nose at the south-eastern corner of the Hoe. This location was further strengthened by the building of a ...",In what year did Parliament pass a notable law that led to the building of fortifications in Plymouth?
1,The Spanish and French were the ones who established early settlements in Florida.,factual,"""By May 1539, Conquistador Hernando de Soto skirted the coast of Florida, searching for a deep harbor to land. He described seeing a thick wall of red mangroves spread mile after mile, some reaching as high as 70 feet (21 m), with intertwined and elevated roots making landing difficult. Very soon, 'many smokes' appeared 'along the whole coast', billowing against the sky, when the Native ancestors of the Seminole spotted the newcomers and spread the alarm by signal fires"". The Spanish introduced Christianity, cattle, horses, sheep, the Spanish language, and more to Florida.[full citation needed] Both the Spanish and French established settlements in Florida, with varying degrees of success. In 1559, Don Tristán de Luna y Arellano established a colony at present-day Pensacola, one of the first European settlements in the continental United States, but it was abandoned by 1561.",Who established early settlements in Florida
2,"Traditionally, monsoons in Punjab are expected to begin in May.",factual,"The onset of the southwest monsoon is anticipated to reach Punjab by May, but since the early 1970s the weather pattern has been irregular. The spring monsoon has either skipped over the area or has caused it to rain so hard that floods have resulted. June and July are oppressively hot. Although official estimates rarely place the temperature above 46 °C, newspaper sources claim that it reaches 51 °C and regularly carry reports about people who have succumbed to the heat. Heat records were broken in Multan in June 1993, when the mercury was reported to have risen to 54 °C. In August the oppressive heat is punctuated by the rainy season, referred to as barsat, which brings relief in its wake. The hardest part of the summer is then over, but cooler weather does not come until late October.",When do monsoons traditionally happen in Punjab?
3,The media made the requests for Kondo to use orchestral music throughout the game.,factual,"Media requests at the trade show prompted Kondo to consider using orchestral music for the other tracks in the game as well, a notion reinforced by his preference for live instruments. He originally envisioned a full 50-person orchestra for action sequences and a string quartet for more ""lyrical moments"", though the final product used sequenced music instead. Kondo later cited the lack of interactivity that comes with orchestral music as one of the main reasons for the decision. Both six- and seven-track versions of the game's soundtrack were released on November 19, 2006, as part of a Nintendo Power promotion and bundled with replicas of the Master Sword and the Hylian Shield.",Who

# Groups / Topics

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from preprocessing import preprocess

In [8]:
# Apply preprocessing
train_df["processed_question"] = train_df["question"].apply(preprocess)
train_df.head()

,answer,type,context,question,processed_question
0,"In 1512, Parliament passed a significant act that resulted in the construction of fortifications in Plymouth.",factual,"During the Hundred Years' War a French attack (1340) burned a manor house and took some prisoners, but failed to get into the town. In 1403 the town was burned by Breton raiders. In the late fifteenth century a 'castle quadrate' was constructed close to the area now known as The Barbican; it included four round towers, one at each corner, as featured on the city coat of arms. The castle served to protect Sutton Pool, which is where the fleet was based in Plymouth prior to the establishment of Plymouth Dockyard. In 1512 an Act of Parliament was passed for further fortifying Plymouth, and a series of fortifications were then built, including defensive walls at the entrance to Sutton Pool (across which a chain would be extended in time of danger). Defences on St Nicholas Island also date from this time, and a string of six artillery blockhouses were built, including one on Fishers Nose at the south-eastern corner of the Hoe. This location was further strengthened by the building of a ...",In what year did Parliament pass a notable law that led to the building of fortifications in Plymouth?,year parliament pass notable law led building fortifications plymouth
1,The Spanish and French were the ones who established early settlements in Florida.,factual,"""By May 1539, Conquistador Hernando de Soto skirted the coast of Florida, searching for a deep harbor to land. He described seeing a thick wall of red mangroves spread mile after mile, some reaching as high as 70 feet (21 m), with intertwined and elevated roots making landing difficult. Very soon, 'many smokes' appeared 'along the whole coast', billowing against the sky, when the Native ancestors of the Seminole spotted the newcomers and spread the alarm by signal fires"". The Spanish introduced Christianity, cattle, horses, sheep, the Spanish language, and more to Florida.[full citation needed] Both the Spanish and French established settlements in Florida, with varying degrees of success. In 1559, Don Tristán de Luna y Arellano established a colony at present-day Pensacola, one of the first European settlements in the continental United States, but it was abandoned by 1561.",Who established early settlements in Florida,established early settlements florida
2,"Traditionally, monsoons in Punjab are expected to begin in May.",factual,"The onset of the southwest monsoon is anticipated to reach Punjab by May, but since the early 1970s the weather pattern has been irregular. The spring monsoon has either skipped over the area or has caused it to rain so hard that floods have resulted. June and July are oppressively hot. Although official estimates rarely place the temperature above 46 °C, newspaper sources claim that it reaches 51 °C and regularly carry reports about people who have succumbed to the heat. Heat records were broken in Multan in June 1993, when the mercury was reported to have risen to 54 °C. In August the oppressive heat is punctuated by the rainy season, referred to as barsat, which brings relief in its wake. The hardest part of the summer is then over, but cooler weather does not come until late October.",When do monsoons traditionally happen in Punjab?,monsoons traditionally happen punjab
3,The media made the requests for Kondo to use orchestral music throughout the game.,factual,"Media requests at the trade show prompted Kondo to consider using orchestral music for the other tracks in the game as well, a notion reinforced by his preference for live instruments. He originally envisioned a full 50-person orchestra for action sequences and a string quartet for more ""lyrical moments"", though the final product used sequenced music instead. Kondo later cited the lack of interactivity that comes with orchestral music as one of the main reasons for the decision. Both six- and seven-track versions of the g

In [9]:
# Vectorize
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(train_df["processed_question"])

In [17]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 57808 stored elements and shape (21021, 1000)>

In [18]:
# Clustering
num_clusters = 20
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
train_df["topic"] = kmeans.fit_predict(X)
train_df.head()

,answer,type,context,question,processed_question,topic
0,"In 1512, Parliament passed a significant act that resulted in the construction of fortifications in Plymouth.",factual,"During the Hundred Years' War a French attack (1340) burned a manor house and took some prisoners, but failed to get into the town. In 1403 the town was burned by Breton raiders. In the late fifteenth century a 'castle quadrate' was constructed close to the area now known as The Barbican; it included four round towers, one at each corner, as featured on the city coat of arms. The castle served to protect Sutton Pool, which is where the fleet was based in Plymouth prior to the establishment of Plymouth Dockyard. In 1512 an Act of Parliament was passed for further fortifying Plymouth, and a series of fortifications were then built, including defensive walls at the entrance to Sutton Pool (across which a chain would be extended in time of danger). Defences on St Nicholas Island also date from this time, and a string of six artillery blockhouses were built, including one on Fishers Nose at the south-eastern corner of the Hoe. This location was further strengthened by the building of a ...",In what year did Parliament pass a notable law that led to the building of fortifications in Plymouth?,year parliament pass notable law led building fortifications plymouth,17
1,The Spanish and French were the ones who established early settlements in Florida.,factual,"""By May 1539, Conquistador Hernando de Soto skirted the coast of Florida, searching for a deep harbor to land. He described seeing a thick wall of red mangroves spread mile after mile, some reaching as high as 70 feet (21 m), with intertwined and elevated roots making landing difficult. Very soon, 'many smokes' appeared 'along the whole coast', billowing against the sky, when the Native ancestors of the Seminole spotted the newcomers and spread the alarm by signal fires"". The Spanish introduced Christianity, cattle, horses, sheep, the Spanish language, and more to Florida.[full citation needed] Both the Spanish and French established settlements in Florida, with varying degrees of success. In 1559, Don Tristán de Luna y Arellano established a colony at present-day Pensacola, one of the first European settlements in the continental United States, but it was abandoned by 1561.",Who established early settlements in Florida,established early settlements florida,9
2,"Traditionally, monsoons in Punjab are expected to begin in May.",factual,"The onset of the southwest monsoon is anticipated to reach Punjab by May, but since the early 1970s the weather pattern has been irregular. The spring monsoon has either skipped over the area or has caused it to rain so hard that floods have resulted. June and July are oppressively hot. Although official estimates rarely place the temperature above 46 °C, newspaper sources claim that it reaches 51 °C and regularly carry reports about people who have succumbed to the heat. Heat records were broken in Multan in June 1993, when the mercury was reported to have risen to 54 °C. In August the oppressive heat is punctuated by the rainy season, referred to as barsat, which brings relief in its wake. The hardest part of the summer is then over, but cooler weather does not come until late October.",When do monsoons traditionally happen in Punjab?,monsoons traditionally happen punjab,9
3,The media made the requests for Kondo to use orchestral music throughout the game.,factual,"Media requests at the trade show prompted Kondo to consider using orchestral music for the other tracks in the game as well, a notion reinforced by his preference for live instruments. He originally envisioned a full 50-person orchestra for action sequences and a string quartet for more ""lyrical moments"", though the final product used sequenced music instead. Kondo later cited the lack of interactivity that comes with orchestral music as one of the main reasons for the decision. Both six- and seven-track vers

In [ ]:
# Group by topic and display sample questions
grouped = train_df.groupby("topic")["question"].apply(list)
for topic, questions in grouped.items():
    print(f"Topic {topic} ({len(questions)} questions)")
    for question in questions[:3]:
        print(f" Q: {question}")

Topic 0 (232 questions)
 Q: Along with anklets, what pieces of jewelry are traditionally worn by Somali women?
 Q: In the Imperial era, what cult did legionnaires follow?
Topic 1 (337 questions)
 Q: Did all the people want him as Patriarch?
 Q: The opposing view states that time is an intellectual concept that allows people to what?
Topic 2 (663 questions)
 Q: In what year was the Presbyterianism church formed in England?
 Q: How much does Barcelona donate to UNICEF per year?
Topic 3 (535 questions)
 Q: In what year was the first black Baptist minister elected to the city council?
 Q: Which financial institution was the first one visible to run into trouble in the United States?
Topic 4 (46 questions)
 Q: What is the main reason that countries were excluded from the 2011 report?
 Q: Where did the New York times report say towers with guns should go?
Topic 5 (445 questions)
 Q: Who used Thracians and Agrianes as light cavalry?
 Q: For what purpose was the Tower first used?
Topic 6 (179 

In [ ]:
# Optional: Visualize clusters in 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.toarray())
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=train_df["topic"], cmap="viridis", alpha=0.5)
plt.title("Question Topics Clustering (PCA 2D)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(label="Topic")
plt.show()
